In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from pathlib import Path
BASE_DIR = Path().resolve().parent   
DATA_DIR = BASE_DIR / "data"

In [12]:
def show_full(df):
    """
    Fully display a DataFrame without any truncation.
    Works for any df[...] slice.
    """
    import pandas as pd
    from IPython.display import display

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", None
    ):
        display(df)

### 0. Read data 

In [ ]:
df = pd.read_parquet('https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet', engine='pyarrow')
print(df.shape)
cols = ['url', 'date', 'language', 'title', 'text']
print(cols)

In [ ]:
# Show a short preview to understand what the raw text looks like
ex = df[['title', 'text']].dropna().sample(5, random_state=42)
for i, (idx, row) in enumerate(ex.iterrows(), 1):
    print("\n" + "="*80)
    print(f"[Example {i}]")
    print(f"Title: {row['title']}")
    print(f"Text preview (first 500 chars):\n{row['text'][:500]}")

In [ ]:
# Analyze text length distribution
text_lengths = df['text'].dropna().astype(str).apply(len)
print("Text length distribution (characters):")
print(text_lengths.describe())

plt.figure(figsize=(12, 2))
text_lengths.plot(kind='box', vert=False)

p = 0.01
q = text_lengths.quantile(p)
print(f"{p*100}% percentile length:", q)

In [ ]:
# Analyze title length distribution
print(df['title'].dropna().astype(str).apply(len).describe())
df['title'].dropna().astype(str).apply(len).quantile(0.99)

In [ ]:
df['text'].isna().sum()

In [ ]:
df['language'].value_counts()

### The lightest and structure-preserving normalization

In [ ]:
def normalize_whitespace(text: str) -> str:
    """
    Light normalization that preserves structure (paragraph boundaries).
    """

    # 1) Normalize all newline styles to '\n'
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2) Collapse multiple spaces or tabs into a single space
    text = re.sub(r"[ \t]+", " ", text)

    # 3) Collapse excessive blank lines (3 or more) into double newline
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 4) Trim leading and trailing whitespace
    text = text.strip()

    return text

In [ ]:
df0 = df.copy()
df0["text_norm0"] = df["text"].astype(str).map(normalize_whitespace)
df0["text_norm0_lengths"] = df0["text_norm0"].astype(str).apply(len)

In [ ]:
# Identify rows where normalization changed the text
mask_changed = df0["text"] != df0["text_norm0"]

print("Changed ratio:", mask_changed.mean())
print("Changed count:", mask_changed.sum())

### Remove docs where length <= 1000
I read through those text, they are mostly just title + boilerplate. That text ≈ title provides almost no info from my perspective.

Even if some normal text are wrongly deleted, it won't affect a lot due to the proportion.

In [ ]:
df1 = df0.copy().query("text_norm0_lengths > 1000")

print("Before:", len(df0), "After:", len(df1))

In [ ]:
# Check docs where title = text
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# --- prepare strings ---
df1["title_clean"] = df1["title"].fillna("").astype(str)
df1["text_clean"]  = df1["text_norm0"].fillna("").astype(str)

# --- restrict to short docs only (speed gate) ---
short_mask = df1["text_clean"].str.len() < 5000

# pre-allocate sim_ratio with NaN for all rows
df1["sim_ratio"] = np.nan

# compute similarity only for short docs
titles = df1.loc[short_mask, "title_clean"].tolist()
texts  = df1.loc[short_mask, "text_clean"].tolist()

df1.loc[short_mask, "sim_ratio"] = [similarity(a, b) for a, b in zip(titles, texts)]

# len_diff is cheap, you can compute for all rows
df1["len_diff"] = (df1["title_clean"].str.len() - df1["text_clean"].str.len()).abs()

# --- define duplicate mask ---
dup_mask = (
    (df1["title_clean"] == df1["text_clean"]) |
    (short_mask & (
        ((df1["len_diff"] <= 20) & (df1["sim_ratio"] > 0.85)) |
        (df1["sim_ratio"] > 0.9)
    ))
)

dup_rows = df1.loc[dup_mask]

print("Near-duplicate rows:", len(dup_rows))
dup_rows.head(2)

### Remove HTML artifacts 

In [ ]:
df2 = df1[~dup_mask].copy()[cols + ["text_norm0", "text_norm0_lengths"]]


In [ ]:
# --- HTML artifact signals ---
tag_re    = re.compile(r"<[^>]+>")        # matches <p>...</p>, <br>, <div ...>, etc.
entity_re = re.compile(r"&[a-zA-Z]+;|&#\d+;")  # matches &amp; &nbsp; &#39; etc.

# --- Count number of HTML tags per article ---
df_sample = df2["text_norm0"].dropna().astype(str).sample(5000, random_state=42).copy()

df_sample_tag_count = df_sample.apply(lambda x: len(tag_re.findall(x)))
print("== tag count ==")
print(df_sample_tag_count.describe())

df_sample_entity_count = df_sample.apply(lambda x: len(entity_re.findall(x)))
print("\n== entity count ==")
print(df_sample_entity_count.describe())

Profiling shows that over 75% of documents contain no HTML tags or entities. The average HTML density is near zero, indicating that the dataset is largely pre-cleaned. Therefore, we apply a uniform HTML stripping procedure for consistency.

> Regex is the "deletion mode" at the string level

> BeautifulSoup is a structure-level "HTML parsing" service

In [ ]:
from bs4 import BeautifulSoup
import html

def clean_html_fast(text):
    if not tag_re.search(text) and not entity_re.search(text):
        return text  # no HTML, skip expensive parsing

    text = html.unescape(text)
    soup = BeautifulSoup(text, "lxml")
    return soup.get_text(" ", strip=True)

# Apply to full dataset
df2["text_clean_v1"] = df2["text_norm0"].astype(str).apply(clean_html_fast)

In [ ]:
df3 = df2.copy()

save_path_v1 = DATA_DIR / "temp" / "text_clean_v1.parquet"
df3.to_parquet(save_path_v1, index=False)

### Irrelevant crawl artifacts

In [2]:
save_path_v1 = DATA_DIR / "temp" / "text_clean_v1.parquet"
df3 = pd.read_parquet(save_path_v1)
df3.head(2)

,url,date,language,title,text,text_norm0,text_norm0_lengths,text_clean_v1
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",3500,"Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,This AI video of gymnastics might be the freak...,4924,This AI video of gymnastics might be the freak...


#### Quick boilerplate filter on overall level

In [7]:
import re
from collections import Counter
from typing import Dict, Any, List, Tuple

# -----------------------------
# 0) Regex building blocks
# -----------------------------
URL_RE = re.compile(r"https?://\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")

def _normalize_newlines(text: str) -> str:
    # handle literal "\n" from crawling dumps
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")
    return text

def _safe_str(x) -> str:
    if x is None:
        return ""
    try:
        return str(x)
    except Exception:
        return ""

def _match_count(patterns: List[re.Pattern], text: str) -> Tuple[int, List[str]]:
    hits = []
    for p in patterns:
        for m in p.finditer(text):
            hits.append(m.group(0))
    return len(hits), hits[:15]  # cap evidence list

def _line_stats(text: str) -> Dict[str, float]:
    lines = [ln.strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]  # non-empty
    if not lines:
        return {"n_lines": 0, "avg_line_len": 0.0, "short_line_ratio": 0.0, "unique_line_ratio": 0.0}
    lens = [len(ln) for ln in lines]
    short_line_ratio = sum(l <= 25 for l in lens) / len(lens)
    unique_line_ratio = len(set(lines)) / len(lines)
    return {
        "n_lines": float(len(lines)),
        "avg_line_len": float(sum(lens) / len(lens)),
        "short_line_ratio": float(short_line_ratio),
        "unique_line_ratio": float(unique_line_ratio),
    }

def _tokenize_simple(text: str) -> List[str]:
    # cheap tokenization for keyword density
    return re.findall(r"[A-Za-z']+", text.lower())

def _keyword_density(tokens: List[str], keyword_set: set) -> float:
    if not tokens:
        return 0.0
    c = sum(1 for t in tokens if t in keyword_set)
    return c / len(tokens)

def _clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))

def _score_from_count(count: int, soft: int, hard: int) -> float:
    """
    Map a count -> [0,1] by soft/hard thresholds.
    <=soft -> 0..~0.4, >=hard -> 1.0
    """
    if count <= 0:
        return 0.0
    if count >= hard:
        return 1.0
    if count <= soft:
        return 0.4 * (count / soft)
    # between soft and hard: ramp 0.4 -> 1.0
    return 0.4 + 0.6 * ((count - soft) / (hard - soft))


# -----------------------------
# 1) Patterns for 5 pollution categories
# -----------------------------

# (1) Navigation / menu / account / edition blocks
NAV_PATTERNS = [
    re.compile(r"\b(sign in|log in|logout|my account|settings)\b", re.IGNORECASE),
    re.compile(r"\b(home|about|contact|careers|faq|apps)\b", re.IGNORECASE),
    re.compile(r"\b(edition|international|arabic|español|espanol)\b", re.IGNORECASE),
    re.compile(r"\b(newsletters?|e-?paper|subscribe|sign up)\b", re.IGNORECASE),
    re.compile(r"\b(topics you follow|follow us)\b", re.IGNORECASE),
    re.compile(r"\b(markets|tech|media|calculators|videos|live tv)\b", re.IGNORECASE),
    re.compile(r"\b(privacy policy|terms|disclaimer|cookie|gdpr)\b", re.IGNORECASE),
    re.compile(r"\b(menu|toggle|×|close icon)\b", re.IGNORECASE),
]

# (2) Ads / marketing injection
AD_PATTERNS = [
    re.compile(r"\b(advertisement|ADVERTISEMENT|sponsored|promoted)\b", re.IGNORECASE),
    re.compile(r"\b(ad feedback|ad choices|ad never loaded|ad prevented)\b", re.IGNORECASE),
    re.compile(r"\b(1xbet|parimatch|mostbet|stake|betting|signup bonus)\b", re.IGNORECASE),
    re.compile(r"\b(microsoft clarity|doubleclick|taboola|outbrain)\b", re.IGNORECASE),
]

# (3) Comments / forms / input validation
COMMENT_PATTERNS = [
    re.compile(r"\b(leave a reply|cancel reply|comments?)\b", re.IGNORECASE),
    re.compile(r"\b(your email address will not be published|required fields)\b", re.IGNORECASE),
    re.compile(r"\b(this field is required|please enter a valid email)\b", re.IGNORECASE),
    re.compile(r"\b(save my name, email, and website)\b", re.IGNORECASE),
    re.compile(r"\b(submit|thank you|close)\b", re.IGNORECASE),
]

# (4) Related / popular / trending / sidebar modules
RELATED_PATTERNS = [
    re.compile(r"\b(most popular|top news|latest news|trending)\b", re.IGNORECASE),
    re.compile(r"\b(related stories|you might also like|read more|explainers?)\b", re.IGNORECASE),
    re.compile(r"\b(article continues below)\b", re.IGNORECASE),
    re.compile(r"\b(recent posts|categories)\b", re.IGNORECASE),
]

# (5) Directory/listing/non-article page signals (Product Hunt / tag pages / archive)
DIRECTORY_PATTERNS = [
    re.compile(r"\b(products|collections|marketplace|launch archive|coming soon)\b", re.IGNORECASE),
    re.compile(r"\b(upvote|comments?\s*\d+|day rank|week rank|featured on)\b", re.IGNORECASE),
    re.compile(r"\b(browse products|topics|jobs|post a job|advertise)\b", re.IGNORECASE),
    re.compile(r"\b(story's credibility|about this launch|makers of)\b", re.IGNORECASE),
]

# Optional: Context-like “Viewing:” rail
VIEWING_PATTERNS = [
    re.compile(r"\bViewing:\b", re.IGNORECASE),
    re.compile(r"\bShare\b.*\bTweet\b.*\bPost\b.*\bEmail\b", re.IGNORECASE | re.DOTALL),
]

# Keyword sets for density checks
NAV_KEYWORDS = {
    "home","about","contact","careers","apps","faq","edition","international","markets","tech","media",
    "videos","subscribe","newsletter","login","logout","signin","signup","privacy","terms","cookie"
}
AD_KEYWORDS = {"advertisement","sponsored","promoted","betting","bonus","ad"}
COMMENT_KEYWORDS = {"comment","comments","reply","submit","required","email"}
RELATED_KEYWORDS = {"related","popular","trending","latest","stories","read","more","explained","explainer"}
DIRECTORY_KEYWORDS = {"products","collections","marketplace","launch","upvote","featured","rank","makers","archive"}


# -----------------------------
# 2) Main scoring function
# -----------------------------
def boilerplate_score(text: str, return_clean_suggestion: bool = True) -> Dict[str, Any]:
    """
    Score how likely `text` is web-crawling boilerplate / non-article junk.

    Covers 5 pollution categories:
      1) navigation/menu blocks
      2) ads/marketing blocks
      3) comments/forms blocks
      4) related/popular/sidebar blocks
      5) directory/listing pages

    Returns:
      {
        "score": 0..1,
        "components": {...},
        "evidence": {...},
        "features": {...},
        "decision": {...}
      }
    """
    raw = _safe_str(text)
    t = _normalize_newlines(raw).strip()

    if not t:
        return {
            "score": 1.0,
            "components": {k: 1.0 for k in ["nav","ads","comments","related","directory"]},
            "evidence": {"reason": ["empty text"]},
            "features": {"len": 0},
            "decision": {"label": "boilerplate", "confidence": "high", "suggest": "drop"},
        }

    # Basic features
    n_chars = len(t)
    urls = URL_RE.findall(t)
    url_chars = sum(len(u) for u in urls)
    url_ratio = (url_chars / n_chars) if n_chars else 0.0
    email_cnt = len(EMAIL_RE.findall(t))
    stats = _line_stats(t)
    tokens = _tokenize_simple(t)

    # Pattern hit counts + evidence snippets
    nav_cnt, nav_hits = _match_count(NAV_PATTERNS, t)
    ad_cnt, ad_hits = _match_count(AD_PATTERNS, t)
    com_cnt, com_hits = _match_count(COMMENT_PATTERNS, t)
    rel_cnt, rel_hits = _match_count(RELATED_PATTERNS, t)
    dir_cnt, dir_hits = _match_count(DIRECTORY_PATTERNS, t)
    view_cnt, view_hits = _match_count(VIEWING_PATTERNS, t)

    # Density signals (helps with "menu word soup" pages)
    nav_density = _keyword_density(tokens, NAV_KEYWORDS)
    ad_density = _keyword_density(tokens, AD_KEYWORDS)
    com_density = _keyword_density(tokens, COMMENT_KEYWORDS)
    rel_density = _keyword_density(tokens, RELATED_KEYWORDS)
    dir_density = _keyword_density(tokens, DIRECTORY_KEYWORDS)

    # Component scoring (each -> [0,1])
    # counts thresholds are tuned for typical news-crawl dumps; tweak if needed
    s_nav = _score_from_count(nav_cnt, soft=3, hard=12)
    s_ads = _score_from_count(ad_cnt, soft=1, hard=5)
    s_com = _score_from_count(com_cnt, soft=1, hard=6)
    s_rel = _score_from_count(rel_cnt, soft=1, hard=6)
    s_dir = _score_from_count(dir_cnt, soft=1, hard=6)

    # Add density & structural boosters (clamped)
    # URL ratio: in real article usually tiny; if >2% it's suspicious; >8% very suspicious
    s_url = _clamp01((url_ratio - 0.02) / 0.06)  # 0 at 2%, 1 at 8%
    # Many short lines suggests menus/rails
    s_shortlines = _clamp01((stats["short_line_ratio"] - 0.25) / 0.45)  # 0 at 25%, 1 at 70%
    # very low unique-line ratio can indicate repeated rails; but also legitimate repeated lines, so mild
    s_repetition = _clamp01((0.85 - stats["unique_line_ratio"]) / 0.35)

    # density boosters (small weights)
    s_nav = _clamp01(s_nav + 0.8 * nav_density)
    s_ads = _clamp01(s_ads + 1.2 * ad_density)
    s_com = _clamp01(s_com + 1.0 * com_density)
    s_rel = _clamp01(s_rel + 0.8 * rel_density)
    s_dir = _clamp01(s_dir + 1.0 * dir_density)

    # Viewing/share rail is a strong indicator; treat as additive kicker
    s_view = _clamp01(view_cnt / 2.0)  # 0, 0.5, 1

    # Weighted total score (interpretable weights)
    # - navigation + directory are most common "non-article" predictors
    # - url_ratio + shortlines capture DOM dumps
    total = (
        0.22 * s_nav +
        0.18 * s_dir +
        0.16 * s_ads +
        0.14 * s_com +
        0.10 * s_rel +
        0.10 * s_url +
        0.06 * s_shortlines +
        0.04 * s_repetition
    )
    total = _clamp01(total + 0.10 * s_view)

    # Decision policy (simple, adjustable)
    # - high score => likely boilerplate page, drop
    # - mid score => keep but try cleaning rules
    # - low score => treat as content
    if total >= 0.70:
        label, conf, suggest = "boilerplate", "high", "drop_or_recrawl"
    elif total >= 0.45:
        label, conf, suggest = "mixed", "medium", "clean_then_keep"
    else:
        label, conf, suggest = "content", "medium" if total >= 0.25 else "high", "keep"

    out = {
        "score": float(total),
        "components": {
            "nav": float(s_nav),
            "ads": float(s_ads),
            "comments": float(s_com),
            "related": float(s_rel),
            "directory": float(s_dir),
            "url": float(s_url),
            "shortlines": float(s_shortlines),
            "repetition": float(s_repetition),
            "viewing_rail": float(s_view),
        },
        "evidence": {
            "nav_hits": nav_hits,
            "ad_hits": ad_hits,
            "comment_hits": com_hits,
            "related_hits": rel_hits,
            "directory_hits": dir_hits,
            "viewing_hits": view_hits,
        },
        "features": {
            "len_chars": int(n_chars),
            "n_urls": int(len(urls)),
            "url_ratio": float(url_ratio),
            "email_cnt": int(email_cnt),
            **stats,
            "nav_density": float(nav_density),
            "ad_density": float(ad_density),
            "comment_density": float(com_density),
            "related_density": float(rel_density),
            "directory_density": float(dir_density),
            "pattern_counts": {
                "nav": int(nav_cnt),
                "ads": int(ad_cnt),
                "comments": int(com_cnt),
                "related": int(rel_cnt),
                "directory": int(dir_cnt),
                "viewing": int(view_cnt),
            },
        },
        "decision": {"label": label, "confidence": conf, "suggest": suggest},
    }

    # Optional: a very lightweight cleaning suggestion (NOT performing cleaning here)
    if return_clean_suggestion:
        out["decision"]["recommended_actions"] = []
        if s_ads >= 0.6:
            out["decision"]["recommended_actions"].append("strip_ad_blocks")
        if s_nav >= 0.6:
            out["decision"]["recommended_actions"].append("strip_nav_footer")
        if s_com >= 0.6:
            out["decision"]["recommended_actions"].append("strip_comments_forms")
        if s_rel >= 0.6:
            out["decision"]["recommended_actions"].append("strip_related_modules")
        if s_dir >= 0.6:
            out["decision"]["recommended_actions"].append("drop_directory_pages")
        if s_url >= 0.6:
            out["decision"]["recommended_actions"].append("filter_high_url_density")
        if s_view >= 0.5:
            out["decision"]["recommended_actions"].append("strip_viewing_share_rail")

    return out

def boilerplate_score_only(text):
    return boilerplate_score(text, return_clean_suggestion=False)["score"]

In [ ]:
# # 1) get score + label
# tmp = df1["text_norm0"].fillna("").astype(str).apply(boilerplate_score)

# df1["bp_score"] = tmp.apply(lambda d: d["score"])
# df1["bp_label"] = tmp.apply(lambda d: d["decision"]["label"])
# df1["bp_suggest"] = tmp.apply(lambda d: d["decision"]["suggest"])

# # 2)  top junk
# df1.sort_values("bp_score", ascending=False)[
#     ["url", "title_clean", "text_norm0_lengths", "bp_score", "bp_label", "bp_suggest"]
# ].head(20)

# only get score for faster processing 
df3["bp_score"] = df3["text_clean_v1"].apply(boilerplate_score_only)

In [ ]:
df3["bp_label"] = pd.cut(
    df3["bp_score"],
    bins=[-0.01, 0.45, 0.7, 1.0],
    labels=["content", "mixed", "boilerplate"]
)
df3[df3["bp_label"] == "boilerplate"].__len__() # check porpotion

4971

In [ ]:
# Quick qualitative check of "boilerplate" rows
def print_full_rows(df, text_col="text_clean_v1", extra_cols=None, n=5, random_state=42):
    sample_df = df.sample(min(n, len(df)), random_state=random_state)

    for i, (idx, row) in enumerate(sample_df.iterrows(), 1):
        print("\n" + "="*120)
        print(f"[Row {i}] Index: {idx}")
        print("="*120)

        if extra_cols:
            for c in extra_cols:
                print(f"{c}: {row[c]}")
            print("-"*120)

        print(row[text_col])
        print("\n" + "="*120)

print_full_rows(
    df3[df3["bp_label"] == "boilerplate"],
    text_col="text_clean_v1",
    extra_cols=["bp_score"],
    n=5
)


[Row 1] Index: 29895
bp_score: 0.7182458365164247
------------------------------------------------------------------------------------------------------------------------
SHIB Partner Bad Idea AI (BAD) Now Supported by This Second Biggest DEX

Advertisement

AD
Ads

Ads
Ads

Ads

Ads

 

Main navigation
News

Bitcoin (BTC) News

Ethereum (ETH) News

Cardano (ADA) News

Ripple and XRP News

Shiba Inu (SHIB) News

Dogecoin (DOGE) News

Meme Cryptocurrencies

NFT News

Scam Alert

Stories

Interviews

Opinions

Reviews

Price Analysis

Bitcoin (BTC) Price Analysis

Ethereum (ETH) Price Analysis

XRP Price Analysis

Cardano (ADA) Price Analysis

Dogecoin (DOGE) Price Analysis

Shiba Inu (SHIB) Price Analysis

TRON (TRX) Price Analysis

Polygon (MATIC) Price Analysis

Litecoin (LTC) Price Analysis

Solana (SOL) Price Analysis

Guides

Blockchain

Ethereum

Cardano

Polygon

Meme Coins

Stablecoins

NFT

Wallets

PR

Advertise

Press releases

Airdrops

Submit Airdrop

Submit Press Release


In [19]:
df4 = df3[df3["bp_label"] != "boilerplate"].copy()
print("Before:", len(df3), "After:", len(df4))

Before: 191666 After: 186695


In [20]:
save_path_v2 = DATA_DIR / "temp" / "text_clean_v2.parquet"
df4.to_parquet(save_path_v2, index=False)

In [21]:
save_path_v2 = DATA_DIR / "temp" / "text_clean_v2.parquet"
df4 = pd.read_parquet(save_path_v2)
df4.head(2)

,url,date,language,title,text,text_norm0,text_norm0_lengths,text_clean_v1,bp_score,bp_label
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",3500,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",0.090024,content
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,This AI video of gymnastics might be the freak...,4924,This AI video of gymnastics might be the freak...,0.459821,mixed


#### Sign-in specific

In [22]:
# --- Sign-in / login boilerplate pattern ---
signin_re = re.compile(
    r"^(?:\s*)("
    r"sign\s*in|log\s*in|"
    r"create\s+account|"
    r"my\s+account|"
    r"forgot\s+your\s+password|"
    r"password\s+recovery|"
    r"recover\s+your\s+password|"
    r"welcome\s*!?\s*log\s*into\s*your\s*account"
    r")(?:\s*)$",
    flags=re.IGNORECASE
)

def remove_signin_lines(text):
    if not text:
        return text
    
    text = str(text)
    
    # 处理字面量 \n
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")
    
    lines = text.splitlines()
    
    cleaned_lines = [
        ln for ln in lines
        if not signin_re.search(ln.strip())
    ]
    
    return "\n".join(cleaned_lines).strip()

In [23]:
df4["text_clean_v2"] = df4["text_clean_v1"].apply(remove_signin_lines)

In [25]:
def extract_signin_lines(text):
    if not text:
        return []
    
    text = str(text)
    
    # 处理字面量 \n
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")
    
    lines = text.splitlines()
    
    removed = [
        ln.strip()
        for ln in lines
        if signin_re.search(ln.strip())
    ]
    
    return removed

import random

def show_removed_signin(df, text_col="text_clean_v1", n=5, random_state=42):
    random.seed(random_state)
    
    sample_df = df.sample(min(n, len(df)), random_state=random_state)
    
    for i, (idx, row) in enumerate(sample_df.iterrows(), 1):
        removed_lines = extract_signin_lines(row[text_col])
        
        if removed_lines:
            print("\n" + "="*120)
            print(f"[Row {i}] Index: {idx}")
            print("="*120)
            
            for ln in removed_lines:
                print("REMOVED:", ln)
            
            print("="*120)

df_with_signin = df4[
    df4["text_clean_v1"].apply(lambda x: len(extract_signin_lines(x)) > 0)
]

show_removed_signin(df_with_signin, n=5)


[Row 1] Index: 81298
REMOVED: Login

[Row 2] Index: 9072
REMOVED: LOGIN
REMOVED: LOGIN
REMOVED: Log in
REMOVED: Log in

[Row 3] Index: 79114
REMOVED: Log in
REMOVED: My Account
REMOVED: Log in
REMOVED: Log in

[Row 4] Index: 54315
REMOVED: Log in
REMOVED: Log In
REMOVED: Log In
REMOVED: Sign In

[Row 5] Index: 11828
REMOVED: Sign In
REMOVED: Sign In
REMOVED: Sign In


In [32]:
import re

# 你原来的 regex 仍然保留（建议把 login 单词放在这里也行）
signin_re = re.compile(
    r"\b("
    r"sign\s*in|log\s*in|login|"
    r"welcome!\s*log\s*into\s*your\s*account|"
    r"forgot\s+your\s+password|password\s+recovery|recover\s+your\s+password|"
    r"your\s+username|your\s+password|"
    r"my\s+account|create\s+account|register|sign\s*up"
    r")\b",
    flags=re.IGNORECASE
)

# login 类“按钮词”（用于兜底：按钮式行检测）
def has_login_phrase(s: str) -> bool:
    s_lower = s.lower()
    return any(
        phrase in s_lower
        for phrase in [
            "log in",
            "login",
            "sign in",
            "sign up",
            "create account",
            "my account",
            "register",
            "subscribe",
            "logout",
        ]
    )

def is_button_like_line(ln: str) -> bool:
    s = ln.strip()
    if not s:
        return False

    if len(s) > 32:
        return False

    letters = [ch for ch in s if ch.isalpha()]
    if letters:
        upper_ratio = sum(ch.isupper() for ch in letters) / len(letters)
    else:
        upper_ratio = 0.0

    toks = re.findall(r"[A-Za-z]+", s.lower())
    if not toks:
        return False

    short_word_ratio = sum(len(t) <= 6 for t in toks) / len(toks)

    has_login_word = has_login_phrase(s)

    return has_login_word and (upper_ratio >= 0.6 or short_word_ratio >= 0.8)


MAX_DEL_LINE_LEN = 100  # 你要的门槛

def remove_signin_lines(text, max_len=MAX_DEL_LINE_LEN):
    if not text:
        return text

    text = str(text)
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")

    cleaned_lines = []
    for ln in text.splitlines():
        s = ln.strip()

        # 只对短行启用删除
        if len(s) <= max_len and signin_re.search(s):
            continue

        # 如果你还想保留“按钮式检测”，也加同样的长度保护
        if len(s) <= max_len and is_button_like_line(s):
            continue

        cleaned_lines.append(ln)

    return "\n".join(cleaned_lines).strip()

In [36]:
def extract_removed_signin_like_lines(text, max_len=MAX_DEL_LINE_LEN):
    if not text:
        return []

    text = str(text)
    if "\\n" in text and "\n" not in text:
        text = text.replace("\\n", "\n")

    removed = []
    for ln in text.splitlines():
        s = ln.strip()
        if len(s) <= max_len and (signin_re.search(s) or is_button_like_line(s)):
            removed.append(s)
    return removed

In [37]:
df_with = df4[df4["text_clean_v1"].apply(lambda x: len(extract_removed_signin_like_lines(x)) > 0)]
# print_full_rows_clean(df_with.sample(3, random_state=42), text_col="text_clean_v1", extra_cols=None, n=3)

# 只打印被删掉的行
for idx in df_with.sample(5, random_state=42).index:
    print("\n" + "="*100)
    print("Index:", idx)
    for ln in extract_removed_signin_like_lines(df4.loc[idx, "text_clean_v1"]):
        print("REMOVED:", ln)


Index: 182299
REMOVED: KERA News Weekday Update Newsletter Signup
REMOVED: KERA News Weekday Update Newsletter Signup
REMOVED: KERA News Weekday Update Newsletter Signup
REMOVED: KERA News Weekday Update Newsletter Signup
REMOVED: SUBSCRIBE

Index: 127203
REMOVED: Sign Up
REMOVED: Sign up for our Newsletters

Index: 55435
REMOVED: Sign In
REMOVED: Sign In

Index: 113478
REMOVED: Log In
REMOVED: Sign up for email newsletters
REMOVED: Sign Up
REMOVED: Log In
REMOVED: Sign up for email newsletters
REMOVED: Sign Up
REMOVED: Log In
REMOVED: Sign up for email newsletters
REMOVED: Sign Up
REMOVED: Sign Up For Newsletters

Index: 156200
REMOVED: Sign in
REMOVED: Welcome! Log into your account
REMOVED: your username
REMOVED: your password
REMOVED: Forgot your password? Get help
REMOVED: Password recovery
REMOVED: Recover your password


#### Navigation/menu blocks - Specific

#### Language choice - Specific

### Keyword pre-filter — Pre-filter docs that likely aren’t about AI impact on industries.